In [1]:
import numpy as np
import wildfire
from utils.functions import G
import matplotlib.pyplot as plt
from scipy.stats import qmc, norm
import random 
import pickle

In [2]:


kap=0.1
eps = 0.3
upc = 3
alp = 0
q = 1
x_min, x_max = 0, 1
y_min, y_max = 0, 1
t_min, t_max = 0, 1
mu=1.25


def runSolver(physical_parameters):
    wildfire_ = wildfire.Fire(**physical_parameters)
    Nx=64   
    Ny = Nx
    Nt = 5 * Nx**2
    u0=lambda X, Y: np.exp(-((X-0.5)**2 + (Y-0.5)**2)/0.02) 
    b0 = lambda X, Y: np.ones_like(X)
    V=lambda x, y, t: (0*x, 0*y)
    t, X, Y, U, B = wildfire_.solvePDE(Nx, Ny, Nt, u0, b0, V, 'FD', 'RK4', last=True, acc=2, sparse=False)

    SRQ_betamax_num   = np.max(B)
    SRQ_umax_num   = np.max(U)
    return SRQ_umax_num



In [3]:
from tqdm import tqdm
from tqdm.notebook import tqdm
tqdm.pandas()

ne25_samples = np.linspace(0.08, 0.12, 25 + 1)
ne25_samples = np.linspace(0.08, 0.12, 25 + 1)
total_srq=[]
seedval=41
for k in range(0, len(ne25_samples)):
    SRQ50=[]
    print(f"Running for upc={ne25_samples[k]:.4f}")
    sampler50 = qmc.LatinHypercube(d=1, seed=seedval+k)
    sample50= sampler50.random(n=50)
    sample50 = sample50.flatten()
    sample50 = norm.ppf(sample50, loc=1.25, scale=0.25)
    sample50 = np.clip(sample50, 1e-6, None)
    print("Starting simulations...")
    for i in tqdm(range(len(sample50)), desc="Simulated samples", leave=False):
        physical_parameters = {    
            'kap': ne25_samples[k], # diffusion coefficient
            'eps': eps, # inverse of activation energy
            'upc': sample50[i], # u phase change
            'q': q, # reaction heat
            'alp': alp, 
            'x_lim': (x_min, x_max), # x-axis domain 
            'y_lim': (y_min, y_max), # y-axis domain
            't_lim': (t_min, t_max), # time domain
            'components':(True, False, True) #diffusion, convection, reaction
        }
        SRQ50.append(runSolver(physical_parameters)) 
    total_srq.append(SRQ50)
total_srq = np.array(total_srq)
print(total_srq.shape)

Running for upc=0.0800
Starting simulations...


Simulated samples:   0%|          | 0/50 [00:00<?, ?it/s]

/Users/meenuravi/Library/CloudStorage/GoogleDrive-ravimeenu18@gmail.com/My Drive/VerVal/Verification-and-Validation-in-Scientific-Computing/homework5/wildfire.py:15: SyntaxWarning: invalid escape sequence '\k'
  """Wildfire constructor.
/Users/meenuravi/Library/CloudStorage/GoogleDrive-ravimeenu18@gmail.com/My Drive/VerVal/Verification-and-Validation-in-Scientific-Computing/homework5/wildfire.py:78: SyntaxWarning: invalid escape sequence '\m'
  """Solve numerical PDE.
/Users/meenuravi/Library/CloudStorage/GoogleDrive-ravimeenu18@gmail.com/My Drive/VerVal/Verification-and-Validation-in-Scientific-Computing/homework5/wildfire.py:251: SyntaxWarning: invalid escape sequence '\m'
  """Solve numerical PDE.
/Users/meenuravi/Library/CloudStorage/GoogleDrive-ravimeenu18@gmail.com/My Drive/VerVal/Verification-and-Validation-in-Scientific-Computing/homework5/numerical/space/fd.py:82: SyntaxWarning: invalid escape sequence '\D'
  """
/Users/meenuravi/Library/CloudStorage/GoogleDrive-ravimeenu18@gm

KeyboardInterrupt: 